In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from expand_subgraph import ExpandSubgraph
from model_2 import GNN_auto, Projector
import pickle as pkl
import numpy as np
import torch
from load_data import DataLoader
# from loader2 import DataLoader2 as GnnDataLoader
from reasoning_module2_ import ReasoningModule2 as ReasoningModule

In [3]:
from data_utils import id2ent, ent2name, id2rel, rel2id

In [4]:
with open("knowledge_graph/queries/CWQ_sim_queries2.pkl", "rb") as f:
    queries = pkl.load(f)


In [5]:
with open("data_for_CL/llm_description.pkl", "rb") as f:
    llm_description = pkl.load(f)

In [6]:
class Config:
    data_path = 'knowledge_graph/KG_data/FB15k-237-betae'
    seed = 1234
    k_rel = 4 # beams
    k_cands = 120
    depth = 2 # max depth of subgraph
    cands_lim = 1024
    gpu = 0
    fact_ratio = 1.0
    val_num = -1 # how many triples are used as the validate set
    add_manual_edges = False
    remove_1hop_edges = True
    not_shuffle_train = False
    device = "cuda:0"

In [ ]:
args = Config()
loader = DataLoader(args, mode='train')

# loader.shuffle_train()
train_graph = loader.train_graph
train_graph_homo = list(set([(h,t) for (h,r,t) in train_graph]))

args.n_ent = loader.n_ent
args.n_rel = loader.n_rel

544230 0
==> removing 1-hop links...
==> done


In [ ]:
GoG_args = {
    'drop_ratio': 0.4,
}

sampler = ExpandSubgraph(
    args.n_ent, args.n_rel, train_graph_homo, train_graph, args,
    GoG_simulation=True,
    GoG_args=GoG_args
)

In [ ]:
id_query = 29
queries[id_query]

{'query_type': ('e', ('r',)),
 'raw_query': (3767, (65,)),
 'named_query': ('Howard_the_Duck',
  ('-/film/film_distributor/films_distributed./film/film_film_distributor_relationship/film',)),
 'transformed_query': 'Which company distributed the film featuring Howard the Duck?',
 'answers_id': [928],
 'answers': ['Universal_Studios'],
 'natural_query': 'Which company distributed the film featuring Howard the Duck?',
 'gold_path': [[928, 64, 3767]],
 '1_hop_edges': [[928, 64, 3767], [3767, 172, 928]]}

In [ ]:
sampler.assign_query(queries[id_query])
topic_ents = sampler.start_entities
subgraph_data = sampler.sampleSubgraph()
subgraph_edges = subgraph_data[2]
answer_ents = queries[id_query]["answers_id"]

Simulating GoG by removing edges...


In [ ]:
print("Topic entities:", topic_ents)
print("Answer entities:", answer_ents)
print("Subgraph data structure:", type(subgraph_data), len(subgraph_data))
print("Subgraph edges structure:", type(subgraph_edges), len(subgraph_edges))
print("First few edges:", subgraph_edges[:5])
print("Answer present in the subgraph:", any(a in subgraph_data[0] for a in answer_ents))

Topic entities: [3767]
Answer entities: [928]
Subgraph data structure: <class 'tuple'> 3
Subgraph edges structure: <class 'torch.Tensor'> 1768
First few edges: tensor([[ 3767,    62,   844],
        [ 3767,    62,   136],
        [ 3767,    62,   174],
        [ 3767,   272,  1196],
        [ 3767,    62, 12061]])
Answer present in the subgraph: True


In [ ]:
import torch

def find_all_paths(topic_entities, answer_entities, subgraph_edges):
    # Convert tensor to list for iteration
    edges = subgraph_edges.tolist()
    
    # Adjacency list: {head_node: [(tail_node, edge_idx), ...]}
    adj = {}
    for  (h, r, t) in edges:
        if h not in adj:
            adj[h] = []
        adj[h].append((t, r))

    all_paths = []
    target_set = set(answer_entities)

    def dfs(current_node, current_path, visited_edges, visited_entities):
        # Base Case: We reached an answer entity
        if current_node in target_set:
            all_paths.append(list(current_path))
            print(f"Found path: {current_path}")
            return
            # We continue searching in case this entity is a bridge 
            # to another answer, or just return if only one answer per path is needed.

        if current_node not in adj:
            return

        for neighbor, edge_r in adj[current_node]:
            # Calculate inverse edge: even -> idx+1, odd -> idx-1
            inverse_r = edge_r + 1 if edge_r % 2 == 0 else edge_r - 1
            # print(f"Current node: {current_node}, Neighbor: {neighbor}, Edge idx: {edge_idx}, Inverse idx: {inverse_idx}")
            # Check if we've used this edge OR if we've already visited the neighbor node
            if (current_node, edge_r, neighbor) not in visited_edges and neighbor not in visited_entities:
                
                # 1. Mark visits
                # print(edge_idx)
                visited_edges.add((current_node, edge_r, neighbor))
                visited_edges.add((neighbor, inverse_r, current_node))
                visited_entities.add(neighbor)
                current_path.append((current_node, edge_r, neighbor))
                
                # 2. Recurse
                dfs(neighbor, current_path, visited_edges, visited_entities)
                
                # 3. Backtrack (Cleanup for other potential paths)
                current_path.pop()
                visited_entities.remove(neighbor)
                visited_edges.remove((current_node, edge_r, neighbor))
                visited_edges.remove((neighbor, inverse_r, current_node))

    # Start search from each topic entity
    for start_node in topic_entities:
        # Initialize with the starting node already "visited"
        dfs(start_node, [], set(), {start_node})

    if not all_paths:
        print(f"Report: No path found from {topic_entities} to {answer_entities}.")
    else:
        print(f"Success: Found {len(all_paths)} path(s).")
    
    return all_paths

In [ ]:
a= find_all_paths(topic_ents, answer_ents, subgraph_edges)

In [ ]:
subgraph_edges[(subgraph_edges[:,0] == 12651) | (subgraph_edges[:,2] == 12651)]

tensor([], size=(0, 3), dtype=torch.int64)

In [ ]:
# t_train_graph = torch.tensor(train_graph)
for edges in subgraph_edges:
    inv_rel_idx = edges[1] + 1 if edges[1] % 2 == 0 else edges[1] - 1
    inv_edge = torch.tensor([edges[2], inv_rel_idx, edges[0]])
    # Check if inv_edge in subgraph_edges
    if any((subgraph_edges[:,0] == inv_edge[0]) & (subgraph_edges[:,1] == inv_edge[1]) & (subgraph_edges[:,2] == inv_edge[2])):
        # print(f"Missing inverse edge for {edges}: {inv_edge}")
        assert False, f"exist inverse edge for {edges}: {inv_edge}"

    # else:
    #     print("haha")